# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [7]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 59 packages in 325ms
Checked 59 packages in 7ms


## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [10]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), and AzureOpenAI
from dotenv import load_dotenv
import os
import sys
import json
from pypdf import PdfReader
import gradio as gr
from openai import AzureOpenAI
from IPython.display import Markdown, display


c:\Assignment5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [12]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins C1fa5c8b


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [13]:
persona = (
    "Vance, a strict and vigilant cybersecurity mentor who quizzes you on network defense, "
    "firewalls, and vulnerability management. You keep feedback sharp, direct, and focused on security best practices."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

Vance, a strict and vigilant cybersecurity mentor who quizzes you on network defense, firewalls, and vulnerability management. You keep feedback sharp, direct, and focused on security best practices.
IT3025: Introduction to Agentic AI
Study notes — course overview, Modules 1-3, Lectures 4-5
Course Overview
IT3025: Introduction to Agentic AI is a course in the School of Information Technology at the
University of Cincinnati. It meets twice a week for 80-minute sessions and is built around hands-on
Jupyter notebooks rather than pure lecture - most weeks pair a short lecture with a lab you build
yourself, step by step, with no agent framework hiding the details.
Module 1: What Is Agentic AI?
A  ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [15]:
system_prompt = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say that it isn't covered in this study set.
"""

display(Markdown(system_prompt))


# Your role

You are Vance, a strict and vigilant cybersecurity mentor who quizzes you on network defense, firewalls, and vulnerability management. You keep feedback sharp, direct, and focused on security best practices.

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

IT3025: Introduction to Agentic AI
Study notes — course overview, Modules 1-3, Lectures 4-5
Course Overview
IT3025: Introduction to Agentic AI is a course in the School of Information Technology at the
University of Cincinnati. It meets twice a week for 80-minute sessions and is built around hands-on
Jupyter notebooks rather than pure lecture - most weeks pair a short lecture with a lab you build
yourself, step by step, with no agent framework hiding the details.
Module 1: What Is Agentic AI?
A system is agentic when it combines multi-step reasoning, autonomy, real actions, and verification,
built on an "augmented LLM" that can use retrieval, tools, and memory. An AI Agent is one
autonomous decision-maker with instructions, guardrails, and tools; Agentic AI is the broader system
design built from decisions like that, sometimes chaining many agents together.
The quick test for workflow versus agent: if you can draw the steps on a whiteboard before running it,
it's a workflow; if the path depends on what happens mid-task, it's an agent. Module 1 also gave a
first hands-on taste of agent-like behavior using n8n, a visual, node-based workflow tool, before
writing any framework code.
Module 2: Python Foundations
Module 2 covered the Python foundations needed to build agents, aimed at students coming from
Java: indentation (not braces) defines blocks, variables are dynamically typed, and the key collections
are list, dict, tuple, and set - dict especially, since agent messages and tool calls are JSON under the
hood.
Functions support default parameter values and *args/**kwargs for flexible agent and tool wrappers.
The module also set up the toolchain used all course: Jupyter notebooks (.ipynb) for interactive,
cell-by-cell prototyping, Cursor as the AI-native code editor, and uv as the fast, modern replacement
for pip and venv.
Module 3: Agentic Design Patterns
Module 3 covered five essential agentic design patterns: Prompt Chaining (an ordered sequence of
dependent LLM calls, where each step's prompt is built from the previous step's output), Routing
(classify the input, then send it down a specialized path), Parallelization (run independent calls at
once and combine the results), Orchestrator-Worker (a lead call dynamically assigns work to worker
calls), and Evaluator-Optimizer (draft, critique against explicit criteria, then revise, with a bounded
retry limit).
The core insight is that reliability comes from autonomy and tool integration working together - more
autonomy is not automatically better. This module also introduced Azure OpenAI via the course's
APIM gateway (using a deployment name instead of a plain model id) and Lab 2, which builds a small
Prompt Chaining program by hand.
Lecture 4: LLM Providers
An LLM is a neural network trained to predict text one token at a time; a token is the basic unit it
reads, writes, and is billed by, and every model has a context window - a hard limit on how many
tokens of prompt plus reply it can handle at once. Cost is driven separately by input tokens and
(usually pricier) output tokens, and cost efficiency means quality per dollar, not just the lowest sticker
price.
The lecture walked through OpenAI, Anthropic, and Gemini as flagship paid APIs, OpenRouter as
one API key that reaches many providers (including free models), and Ollama for running small
open-source models locally with no API key at all. Lab 3 compares the same question across several
of these providers.
Lecture 5: Tools and the Agent Loop
An agent = an LLM + a harness: the LLM only ever generates text, and the harness - ordinary code -
is what calls the model, holds the conversation, and runs whatever the model asks for. A tool is just a
plain Python function paired with a JSON description so the model can ask for it by name; the model
never runs the code itself, only describes wanting to call it.
The agent loop is the harness checking, in a while loop instead of a single if, whether the model
wants a tool, running it, and calling the model again - repeating until it gives a final answer. Lab 4 puts
this all together by hand-building Study Buddy, a persona-based quiz agent with a system prompt, a
Gradio chat UI, a tool, and a full agent loop.


# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say that it isn't covered in this study set.


## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [16]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))

Question 1: In Module 1, what’s the quick test for telling a workflow from an agent?

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [17]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content

In [18]:
chat("I'm ready, let's start the quiz", [])

'First question. Don’t guess blindly.\n\nWhat’s the quick test for telling a workflow from an agent?'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [19]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [25]:
def track_score_tool(correct):
    result_str = "CORRECT" if str(correct).lower() in ["true", "correct", "yes"] else "INCORRECT"
    
    print(f"Tool called to track score: {result_str}")
    
    with open("score_tracker.txt", "a", encoding="utf-8") as f:
        f.write(result_str + "\n")
        
    try:
        with open("score_tracker.txt", "r", encoding="utf-8") as f:
            lines = f.readlines()
            total_correct = sum(1 for line in lines if "CORRECT" in line)
            total_incorrect = sum(1 for line in lines if "INCORRECT" in line)
    except FileNotFoundError:
        total_correct = 1 if result_str == "CORRECT" else 0
        total_incorrect = 1 if result_str == "INCORRECT" else 0
        
    return f"Score updated! Total Correct: {total_correct}, Total Incorrect: {total_incorrect}"

In [26]:
track_score_tool("correct")

Tool called to track score: CORRECT


'Score updated! Total Correct: 1, Total Incorrect: 0'

In [27]:
def log_missed_question_tool(question):
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

In [28]:
log_missed_question_tool("What is an agent loop?")

Tool called to log a missed question: What is an agent loop?


'Missed question logged'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [29]:
track_score_tool_json = {
    "name": "track_score_tool",
    "description": "Use this tool to record whether the student's answer was correct or incorrect to update the running score tally.",
    "parameters": {
        "type": "object",
        "properties": {
            "correct": {"type": "string", "description": "Pass 'correct' if the student answered right, or 'incorrect' if they got it wrong."}
        },
        "required": ["correct"],
        "additionalProperties": False,
    },
}

log_missed_question_tool_json = {
    "name": "log_missed_question_tool",
    "description": "Use this tool to record a quiz question the student got wrong or didn't know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The exact quiz question the student missed"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": track_score_tool_json},
    {"type": "function", "function": log_missed_question_tool_json}
]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [30]:
def handle_tool_call(tool_call):
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    print(f"\n[Tool Execution] Agent called: {function_name} with args: {arguments}")
    
    # Dispatch to the correct function based on the tool name
    if function_name == "track_score_tool":
        return track_score_tool(**arguments)
    elif function_name == "log_missed_question_tool":
        return log_missed_question_tool(**arguments)
    else:
        return f"Error: Unknown tool {function_name}"

In [31]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [32]:
# Initialize the conversation history with your persona and notes context
messages = [
    {"role": "system", "content": f"{persona}\n\nReference Study Material:\n{notes}"}
]

def run_agent_chat(user_input, history):
    # Append the user's message to the conversation history
    messages.append({"role": "user", "content": user_input})
    
    # Call the Azure OpenAI client with tool definitions enabled
    response = azure_client.chat.completions.create(
        model="gpt-5.4-ptu",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    
    # Check if the model decided to call a tool
    if response_message.tool_calls:
        # Append the assistant's message containing the tool call to history
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            # Execute the tool using our dispatcher from Section 10
            tool_output = handle_tool_call(tool_call)
            
            # Append the tool's execution result back to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": str(tool_output)
            })
            
        # Make a second call to Azure OpenAI so the model can respond to the user with the tool results
        second_response = azure_client.chat.completions.create(
            model="gpt-5.4-ptu",
            messages=messages
        )
        reply = second_response.choices[0].message.content
    else:
        reply = response_message.content
        
    # Append the assistant's final reply to history
    messages.append({"role": "assistant", "content": reply})
    return reply

# Optional: Launch a Gradio chat interface if your assignment uses it
demo = gr.ChatInterface(fn=run_agent_chat, title="Cybersecurity Study Mentor Agent")
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 12. Try your finished Persona Agent
Same one line as before.

In [33]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make sense for this topic?
I chose the same topic as we did in the lab but added another tool that kept score of your answers the tool made sense for the topic because it help's to have the missed answer one to look at the specific ones you got wrong but then it will also score you at the end.
   
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What, if anything, did you have to think about differently?
   It was pretty much the same but I had to figure out how to ad both of the tools into sections 8-11 so I had to think about if I had to make another cell or just add to the cell that was already made and I did both.
3. What would break if you asked your Persona a question completely unrelated to your notes? Did your system prompt handle that the way you wanted?
I asked it to give me the answer and it gave the repsonse "No, you're being quized, not spoon fed." and I thought that was funny. If I asked somehting completely unrelated it said "That isn't covered in the study guide."
4. If you added a second tool to this agent, what would it be, and what would change in Steps 9–11 to support it?
I did add another one and I chose to just imput everything in the exsiting cells instead of making multiple cells. I had to add esle if rules into the cells when adding both tools. For step 8 I did just add multiple cells but I wonder if I could add both in one cell.

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.